# 02 — Data Quality Checks on First-Transaction Data

This notebook checks **which of the data-quality issues found in `01_eda_raw` survive after restricting to each customer's first transaction**. It does *not* clean the data — it only quantifies what remains, so we can decide how to handle each issue before building customer-level features.

Source: `data/processed/first_transaction_churn.csv`, produced by `data/processing/prepare_churn_data.py` (one row per line item of each customer's **first** invoice, plus a `churn` label). The raw figures referenced below come from `01_eda_raw`.

In [61]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

PROCESSED = Path('..') / 'data' / 'processed'
ft = pd.read_csv(PROCESSED / 'first_transaction_churn.csv', parse_dates=['invoice_date'])

n_rows, n_inv, n_cust = len(ft), ft['invoice'].nunique(), ft['customer_id'].nunique()
print(f'Rows: {n_rows:,} | first-transaction invoices: {n_inv:,} | customers: {n_cust:,}')
ft.head()

Rows: 126,080 | first-transaction invoices: 5,346 | customers: 5,346


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,churn
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,0
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,0
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,0


## 1. Missing values (`customer_id`, `description`)

Raw data had 22.77% missing `customer_id` and 0.41% missing `description`. Rows with no `customer_id` are dropped when building the first-transaction table, so this should be 0; we also check whether any first-invoice line still lacks a description.

In [62]:
print('Missing values in first_txn  (raw 01 figure in brackets):')
print(f"  customer_id : {ft['customer_id'].isna().sum():>6,} ({ft['customer_id'].isna().mean()*100:5.2f}%)   [raw: 22.77%]")
print(f"  description : {ft['description'].isna().sum():>6,} ({ft['description'].isna().mean()*100:5.2f}%)   [raw:  0.41%]")

Missing values in first_txn  (raw 01 figure in brackets):
  customer_id :      0 ( 0.00%)   [raw: 22.77%]
  description :      0 ( 0.00%)   [raw:  0.41%]


## 2. Zero prices (`price == 0`)

Raw data had 6,202 rows with `price == 0` (likely freebies / adjustments).

In [63]:
zero_price = ft['price'] == 0
print(f"price == 0 rows : {zero_price.sum():,} ({zero_price.mean()*100:.2f}%)  across {ft.loc[zero_price,'invoice'].nunique():,} invoices   [raw: 6,202 rows]")
ft[zero_price].head(10)

price == 0 rows : 7 (0.01%)  across 7 invoices   [raw: 6,202 rows]


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,churn
3185,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.00,16126,United Kingdom,0.00,0
4137,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.00,15658,United Kingdom,0.00,1
10399,490727,M,Manual,1,2009-12-07 16:38:00,0.00,17231,United Kingdom,0.00,0
17576,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.00,15070,United Kingdom,0.00,1
32314,497819,TEST001,This is a test product.,5,2010-02-12 14:58:00,0.00,14103,United Kingdom,0.00,1
32350,497843,TEST001,This is a test product.,5,2010-02-12 15:47:00,0.00,14827,United Kingdom,0.00,1
79678,524181,46000M,POLYESTER FILLER PAD 45x45cm,648,2010-09-27 16:59:00,0.00,17450,United Kingdom,0.00,0


In [37]:
# pd.set_option('display.max_rows', None)
# print(ft['description'].value_counts(ascending=False).reset_index())

                              description  count
0      WHITE HANGING HEART T-LIGHT HOLDER    886
1           BAKING SET 9 PIECE RETROSPOT     521
2            REX CASH+CARRY JUMBO SHOPPER    400
3           ASSORTED COLOUR BIRD ORNAMENT    391
4                REGENCY CAKESTAND 3 TIER    386
5             60 TEATIME FAIRY CAKE CASES    352
6        PACK OF 72 RETRO SPOT CAKE CASES    340
7          STRAWBERRY CERAMIC TRINKET BOX    331
8      PACK OF 60 PINK PAISLEY CAKE CASES    314
9                      VINTAGE SNAP CARDS    309
10               HOME BUILDING BLOCK WORD    300
11       RED HANGING HEART T-LIGHT HOLDER    290
12               LOVE BUILDING BLOCK WORD    281
13            ZINC METAL HEART DECORATION    274
14                                POSTAGE    273
15             FELTCRAFT 6 FLOWER FRIENDS    273
16     VINTAGE HEADS AND TAILS CARD GAME     257
17        PAPER CHAIN KIT 50'S CHRISTMAS     256
18      RETRO SPOT TEA SET CERAMIC 11 PC     256
19                  

In [40]:
ft[ft['description'] == 'Manual']

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,churn
5767,C490126,M,Manual,-1,2009-12-03 18:12:00,5.95,15884,United Kingdom,-5.95,0
7097,490300,M,Manual,1,2009-12-04 14:19:00,0.85,12970,United Kingdom,0.85,1
7098,490300,M,Manual,1,2009-12-04 14:19:00,0.21,12970,United Kingdom,0.21,1
10399,490727,M,Manual,1,2009-12-07 16:38:00,0.00,17231,United Kingdom,0.00,0
10599,490760,M,Manual,1,2009-12-08 09:49:00,10.00,14295,United Kingdom,10.00,0
11804,490999,M,Manual,1,2009-12-08 17:26:00,15.95,13883,United Kingdom,15.95,0
11805,490999,M,Manual,1,2009-12-08 17:26:00,8.70,13883,United Kingdom,8.70,0
13027,C491157,M,Manual,-1,2009-12-10 10:00:00,40.00,14942,United Kingdom,-40.00,0
21547,C494024,M,Manual,-1,2010-01-11 10:09:00,17.60,12959,United Kingdom,-17.60,1
26397,495409,M,Manual,1,2010-01-25 10:26:00,29.95,17810,United Kingdom,29.95,0


In [39]:
ft.loc[~ft['description'].str.isupper(), 'description'].value_counts()

description
Manual                                 163
BAG 125g SWIRLY MARBLES                 86
BAG 250g SWIRLY MARBLES                 81
POLYESTER FILLER PAD 45x45cm            60
POLYESTER FILLER PAD 40x40cm            60
Adjustment by john on 26/01/2010 16     24
3 TRADITIONAl BISCUIT CUTTERS  SET      18
BAG 500g SWIRLY MARBLES                 16
POLYESTER FILLER PAD 65CMx65CM          12
POLYESTER FILLER PAD 30CMx30CM          11
ESSENTIAL BALM 3.5g TIN IN ENVELOPE     10
POLYESTER FILLER PAD 45x30cm             7
Adjustment by john on 26/01/2010 17      7
Next Day Carriage                        6
FOLK ART GREETING CARD,pack/12           6
FRENCH BLUE METAL DOOR SIGN No           5
This is a test product.                  4
NUMBER TILE COTTAGE GARDEN No            3
Discount                                 2
THE KING GIFT BAG 25x24x12cm             2
 Bank Charges                            2
Bank Charges                             1
FRENCH BLUE METAL DOOR SIGN, No          1

## 3. One stock code → many descriptions

Raw data reported 53.38% of stock codes carrying more than one description — but that count treated *missing* descriptions as a distinct value, so it was inflated. `first_txn` has no missing descriptions (see §1), so this is a cleaner measure of genuinely conflicting labels.

In [5]:
desc_norm = ft['description'].astype(str).str.strip().str.upper()

code_desc = (ft.assign(description_norm=desc_norm)
               .groupby('stock_code')['description_norm']
               .nunique()
               .sort_values(ascending=False)
               .reset_index(name='unique_descriptions'))
n_codes = len(code_desc); n_multi = (code_desc['unique_descriptions'] > 1).sum()
print(f'Stock codes total           : {n_codes:,}')
print(f'Stock codes with >1 desc    : {n_multi:,} ({n_multi/n_codes*100:.2f}%)   [raw: 53.38%, but raw counted missing description as a value]')
code_desc[code_desc['unique_descriptions'] > 1].head(20)

Stock codes total           : 4,188
Stock codes with >1 desc    : 381 (9.10%)   [raw: 53.38%, but raw counted missing description as a value]


,stock_code,unique_descriptions
0,20685,4
1,22785,3
2,22845,3
3,84997C,3
4,84997B,3
5,22333,3
6,84997A,3
7,22356,3
8,22844,3
9,85099B,3


In [6]:
# Check a couple of examples with the stock_ids that have multiple descriptions
for i in ('20685', '22785', '22845', '84997C'):
    print(f'Number of unique descriptions for {i}: \n {ft[ft['stock_code'] == i]['description'].unique()}')

Number of unique descriptions for 20685: 
 ['RED SPOTTY COIR DOORMAT' 'DOOR MAT RED SPOT' 'DOORMAT RED SPOT'
 'DOORMAT RED RETROSPOT']
Number of unique descriptions for 22785: 
 ['CUSHION COVER PINK UNION FLAG' 'SQUARECUSHION COVER PINK UNION FLAG'
 'SQUARECUSHION COVER PINK UNION JACK']
Number of unique descriptions for 22845: 
 ['CAT FOOD CONTAINER , VINTAGE' 'VINTAGE CREAM CAT FOOD CONTAINER'
 'VINTAGE CAT FOOD CONTAINER']
Number of unique descriptions for 84997C: 
 ['BLUE 3 PIECE MINI DOTS CUTLERY SET' 'BLUE 3 PIECE POLKADOT CUTLERY SET'
 'CHILDRENS CUTLERY POLKADOT BLUE']


## 4. One description → many stock codes

Raw data: 5.20% of descriptions mapped to more than one stock code (generic labels like `CHECK`, `DAMAGED`, `?`).

In [7]:
desc_code = (ft.assign(description_norm=desc_norm)
               .groupby('description_norm')['stock_code']
               .nunique()
               .sort_values(ascending=False)
               .reset_index(name='unique_stock_codes'))
n_desc = len(desc_code); n_multi = (desc_code['unique_stock_codes'] > 1).sum()

print(f'Distinct descriptions       : {n_desc:,}')
print(f'Descriptions with >1 code   : {n_multi:,} ({n_multi/n_desc*100:.2f}%)   [raw: 5.20%]')
desc_code[desc_code['unique_stock_codes'] > 1].head(32)

Distinct descriptions       : 4,556
Descriptions with >1 code   : 32 (0.70%)   [raw: 5.20%]


,description_norm,unique_stock_codes
0,COLUMBIAN CANDLE ROUND,4
1,"METAL SIGN,CUPCAKE SINGLE HOOK",3
2,MODERN CHRISTMAS TREE CANDLE,3
3,COLOURING PENCILS BROWN TUBE,3
4,FAIRY CAKE PLACEMATS,2
5,PASTEL BLUE PHOTO ALBUM,2
6,RETRO PLASTIC 70'S TRAY,2
7,RETRO PLASTIC DAISY TRAY,2
8,RETRO PLASTIC POLKA TRAY,2
9,PASTEL PINK PHOTO ALBUM,2


In [13]:
ft[ft['description'] == 'METAL SIGN,CUPCAKE SINGLE HOOK']

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,churn
4607,490017,82613A,"METAL SIGN,CUPCAKE SINGLE HOOK",1,2009-12-03 12:31:00,1.25,14159,United Kingdom,1.25,0
4608,490017,82613B,"METAL SIGN,CUPCAKE SINGLE HOOK",1,2009-12-03 12:31:00,1.25,14159,United Kingdom,1.25,0
4610,490017,82613C,"METAL SIGN,CUPCAKE SINGLE HOOK",1,2009-12-03 12:31:00,1.25,14159,United Kingdom,1.25,0
9010,490503,82613A,"METAL SIGN,CUPCAKE SINGLE HOOK",3,2009-12-06 14:00:00,1.25,14548,United Kingdom,3.75,0
12646,491113,82613C,"METAL SIGN,CUPCAKE SINGLE HOOK",1,2009-12-09 14:48:00,1.25,14505,United Kingdom,1.25,0
12648,491113,82613A,"METAL SIGN,CUPCAKE SINGLE HOOK",1,2009-12-09 14:48:00,1.25,14505,United Kingdom,1.25,0
17966,492246,82613C,"METAL SIGN,CUPCAKE SINGLE HOOK",8,2009-12-16 09:41:00,1.25,13952,United Kingdom,10.00,0
29057,496502,82613A,"METAL SIGN,CUPCAKE SINGLE HOOK",8,2010-02-02 09:39:00,1.25,17146,United Kingdom,10.00,0
29666,496730,82613A,"METAL SIGN,CUPCAKE SINGLE HOOK",8,2010-02-03 14:24:00,1.25,16834,United Kingdom,10.00,1
29667,496730,82613B,"METAL SIGN,CUPCAKE SINGLE HOOK",8,2010-02-03 14:24:00,1.25,16834,United Kingdom,10.00,1


## 5. Stock code format

Issue #5 from `01`: stock codes are stored in different formats — numbers only, letters only (`POST`, `M`, `DOT`, …), number + letters (product variants such as `84031A`), or other (`gift_0001_80`, `BANK CHARGES`). The **letters-only** and **other** formats are not valid product codes and need processing; number + letters are mostly legitimate variants. Here we check whether the *invalid* formats still appear after keeping only the first transaction.

In [14]:
# Format composition of the unique stock codes.
u = pd.Series(ft['stock_code'].astype(str).unique())
is_num   = u.str.fullmatch(r'[0-9]+')
is_alpha = u.str.fullmatch(r'[A-Za-z]+')
is_mix   = u.str.fullmatch(r'(?=.*[A-Za-z])(?=.*[0-9])[A-Za-z0-9]+')
is_other = ~(is_num | is_alpha | is_mix)

n = len(u)
print(f'Unique stock codes        : {n:,}')
print(f'  numeric only            : {is_num.sum():>5,} ({is_num.mean()*100:5.2f}%)')
print(f'  letters only            : {is_alpha.sum():>5,} ({is_alpha.mean()*100:5.2f}%)')
print(f'  number + letters (mixed): {is_mix.sum():>5,} ({is_mix.mean()*100:5.2f}%)')
print(f'  other (symbols/spaces)  : {is_other.sum():>5,} ({is_other.mean()*100:5.2f}%)')

# Letters-only and "other" are NOT valid product codes -> these still need processing.
invalid_codes = set(u[is_alpha | is_other])
invalid_row = ft['stock_code'].astype(str).isin(invalid_codes)
print('\nInvalid-format codes (letters-only or other) still present after the 1st-transaction filter:')
print(f'  unique codes : {sorted(invalid_codes)}')
print(f'  line rows    : {invalid_row.sum():,} ({invalid_row.mean()*100:.2f}%)')
print(f'  invoices     : {ft.loc[invalid_row, "invoice"].nunique():,} ({ft.loc[invalid_row, "invoice"].nunique()/n_inv*100:.2f}%)')

Unique stock codes        : 4,188
  numeric only            : 3,001 (71.66%)
  letters only            :     6 ( 0.14%)
  number + letters (mixed): 1,180 (28.18%)
  other (symbols/spaces)  :     1 ( 0.02%)

Invalid-format codes (letters-only or other) still present after the 1st-transaction filter:
  unique codes : ['ADJUST', 'BANK CHARGES', 'D', 'DOT', 'M', 'PADS', 'POST']
  line rows    : 480 (0.38%)
  invoices     : 452 (8.45%)


In [59]:
ft[(ft['stock_code'].str.fullmatch(r'(?=.*[A-Za-z])(?=.*[0-9])[A-Za-z0-9]+') == True)].head()
   # (ft['stock_code'].str.fullmatch(r'[A-Za-z]+') == False) &
   # (ft['stock_code'].str.fullmatch(r'(?=.*[A-Za-z])(?=.*[0-9])[A-Za-z0-9]+') == False)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,churn
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0
8,489436,48173C,DOOR MAT BLACK FLOCK,10,2009-12-01 09:06:00,5.95,13078,United Kingdom,59.50,0
19,489436,35004B,SET OF 3 BLACK FLYING DUCKS,12,2009-12-01 09:06:00,4.65,13078,United Kingdom,55.80,0
24,489436,84596F,SMALL MARSHMALLOWS PINK BOWL,8,2009-12-01 09:06:00,1.25,13078,United Kingdom,10.00,0


## 6. Cancellations (invoices starting with `C`)

These are **kept by design** in the first-transaction table (a customer whose first invoice is a cancellation is still a customer). We just measure how many there are.

In [16]:
is_cancel = ft['invoice'].astype(str).str.startswith('C')
n_cancel_inv = ft.loc[is_cancel, 'invoice'].nunique()
print(f'Cancellation (C) line rows : {is_cancel.sum():,}')
print(f'Cancellation invoices      : {n_cancel_inv:,} ({n_cancel_inv/n_inv*100:.2f}%)   [kept by design]')

Cancellation (C) line rows : 625
Cancellation invoices      : 274 (5.13%)   [kept by design]


## 7. Outliers in the continuous variables

Tukey 1.5×IQR rule on the positive first-transaction rows (`quantity > 0` & `price > 0`), same definition as `01`'s Section 6.

In [18]:
clean = ft[(ft['quantity'] > 0) & (ft['price'] > 0)].copy()
print('Tukey IQR outliers (positive first-transaction rows):')
for col in ['quantity', 'price', 'line_total']:
    s = clean[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    is_out = (s < lo) | (s > hi)
    print(f'  {col:11s}: Q1={q1:,.2f}  Q3={q3:,.2f}  upper fence={hi:,.2f}  ->  '
          f'{is_out.sum():,} outliers ({is_out.mean()*100:.2f}%)')

Tukey IQR outliers (positive first-transaction rows):
  quantity   : Q1=2.00  Q3=12.00  upper fence=27.00  ->  5,217 outliers (4.16%)
  price      : Q1=1.25  Q3=3.75  upper fence=7.50  ->  10,476 outliers (8.35%)
  line_total : Q1=4.95  Q3=17.70  upper fence=36.83  ->  7,709 outliers (6.15%)


## Summary — what survives the first-transaction filter

| # | Issue | Raw (01) | First transactions | Status |
|---|---|---|---|---|
| 1 | Missing `customer_id` / `description` | 22.77% / 0.41% | 0 / 0 | ✅ resolved |
| 2 | `price = 0` | 6,202 rows | 7 rows (7 invoices) | ⚠️ nearly gone |
| 3 | stock_code → >1 description | 53.38%\* | 9.10% | ⚠️ reduced |
| 4 | description → >1 stock_code | 5.20% | 0.70% | ⚠️ reduced |
| 5 | stock_code format (letters-only / other invalid) | present | 7 invalid codes, 480 rows (0.38%), 452 invoices | ❌ still present, needs processing |
| 6 | cancellations (`C` invoices) | 8,292 invoices | 274 invoices (5.13%) | ❌ kept by design |
| 7 | outliers (IQR) | present | ~4–8% per column | ❌ still present |

\* the raw figure counted missing descriptions as a distinct value, so it was inflated.

**Still to decide handling for, before building customer-level features:** zero prices (2), conflicting descriptions (3 / 4), invalid stock-code formats (5), cancellations (6), and outliers (7). Those decisions feed `data/processing/build_features.py` (next step).